In [8]:
from konlpy.tag import Okt

okt = Okt()

text = "나는 오늘 카페에서 공부했습니다."

print("형태소 : ", okt.morphs(text))
print("명사 : ", okt.nouns(text))
print("품사 : ", okt.pos(text))

형태소 :  ['나', '는', '오늘', '카페', '에서', '공부', '했습니다', '.']
명사 :  ['나', '오늘', '카페', '공부']
품사 :  [('나', 'Noun'), ('는', 'Josa'), ('오늘', 'Noun'), ('카페', 'Noun'), ('에서', 'Josa'), ('공부', 'Noun'), ('했습니다', 'Verb'), ('.', 'Punctuation')]


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

text = "나는 오늘 카페에서 공부했습니다."

result = tokenizer(text)

print(result)

{'input_ids': [101, 100585, 9580, 118762, 9786, 119391, 11489, 8896, 14646, 119424, 119081, 48345, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [17]:
import numpy as np

sentence = "나는 밥을 먹고 싶다"

words = sentence.split()
print("words : ", words)

vocab = list(dict.fromkeys(words))
print()
print("vocab : ", vocab)

word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
print()
print("word2idx : ", word2idx)
print()
print("idx2word : ", idx2word)

sentence_indices = [word2idx[word] for word in words]
print()
print("sentence_indices : ", sentence_indices)

restored_sentence = [idx2word[index] for index in sentence_indices]
print()
print("restored_sentence : ", restored_sentence)


def one_hot(index, vocab_size):
    vector = np.zeros(vocab_size)

    vector[index] = 1

    return vector


vocab_size = len(vocab)
print()
print("vocab_size : ", vocab_size)

one_hot_vectors = []

for index in sentence_indices:
    vector = one_hot(index, vocab_size)
    one_hot_vectors.append(vector)

print()
print("one_hot_vectors : ", one_hot_vectors)

one_hot_vectors = np.array(one_hot_vectors)

X = sentence_indices[:-1]
Y = sentence_indices[1:]

print()
print("X : ", X)
print()
print("y : ", Y)

print()
for x, y in zip(X, Y):
    print(f"{idx2word[x]} -> {idx2word[y]}")

X_one_hot = np.array([one_hot(index, vocab_size) for index in X])
print()
print("X_one_hot : ", X_one_hot)


for i in range(len(X)):
    print(f"\nTime Step {i + 1}")

    print(f"입력 단어 : {idx2word[X[i]]}")
    print(f"입력 숫자 : {X[i]}")
    print(f"입력 벡터 : {X_one_hot[i]}")

    print(f"정답 단어 : {idx2word[Y[i]]}")
    print(f"정답 숫자 : {Y[i]}")

words :  ['나는', '밥을', '먹고', '싶다']

vocab :  ['나는', '밥을', '먹고', '싶다']

word2idx :  {'나는': 0, '밥을': 1, '먹고': 2, '싶다': 3}

idx2word :  {0: '나는', 1: '밥을', 2: '먹고', 3: '싶다'}

sentence_indices :  [0, 1, 2, 3]

restored_sentence :  ['나는', '밥을', '먹고', '싶다']

vocab_size :  4

one_hot_vectors :  [array([1., 0., 0., 0.]), array([0., 1., 0., 0.]), array([0., 0., 1., 0.]), array([0., 0., 0., 1.])]

X :  [0, 1, 2]

y :  [1, 2, 3]

나는 -> 밥을
밥을 -> 먹고
먹고 -> 싶다

X_one_hot :  [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]]

Time Step 1
입력 단어 : 나는
입력 숫자 : 0
입력 벡터 : [1. 0. 0. 0.]
정답 단어 : 밥을
정답 숫자 : 1

Time Step 2
입력 단어 : 밥을
입력 숫자 : 1
입력 벡터 : [0. 1. 0. 0.]
정답 단어 : 먹고
정답 숫자 : 2

Time Step 3
입력 단어 : 먹고
입력 숫자 : 2
입력 벡터 : [0. 0. 1. 0.]
정답 단어 : 싶다
정답 숫자 : 3


In [21]:
import numpy as np

sentence = "나는 밥을 먹고 싶다"
words = sentence.split()
vocab = list(dict.fromkeys(words))

word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for word, i in word2idx.items()}


def one_hot(index, vocab_size):
    vector = np.zeros(vocab_size)

    vector[index] = 1
    return vector


indices = [word2idx[word] for word in words]


X = indices[:-1]
Y = indices[1:]


class RNN:
    def __init__(self, input_size, hidden_size, vocab_size):
        self.W_x = np.random.randn(input_size, hidden_size) * 0.01
        self.W_h = np.random.randn(hidden_size, hidden_size) * 0.01
        self.W_y = np.random.randn(hidden_size, vocab_size) * 0.01
        self.b_h = np.zeros(hidden_size)
        self.b_y = np.zeros(vocab_size)

    def forward(self, x, h_prev):
        h = np.tanh(x @ self.W_x + h_prev @ self.W_h + self.b_h)
        logits = h @ self.W_y + self.b_y

        return h, logits


vocab_size = len(vocab)

hidden_size = 18

rnn = RNN(input_size=vocab_size, hidden_size=hidden_size, vocab_size=vocab_size)

h = np.zeros(hidden_size)

for t in range(len(X)):
    x_word = idx2word[X[t]]

    x = one_hot(X[t], vocab_size)

    h, logits = rnn.forward(x, h)

    predicted_index = np.argmax(logits)

    predicted_word = idx2word[predicted_index]

    target_word = idx2word[Y[t]]

    print("=" * 50)

    print(f"Time Step : {t + 1}")

    print(f"입력       : {x_word}")

    print(f"정답       : {target_word}")

    print(f"현재 예측   : {predicted_word}")

    print()

    print("현재 Hidden State")

    print(h)

Time Step : 1
입력       : 나는
정답       : 밥을
현재 예측   : 싶다

현재 Hidden State
[ 0.00911408 -0.00969136 -0.01539829 -0.00413243 -0.00977661  0.00343389
  0.00141604  0.00223266  0.00579695 -0.00236033 -0.01545094  0.01130533
  0.0011104  -0.01030271 -0.00568741 -0.00033649 -0.00557579  0.00361183]
Time Step : 2
입력       : 밥을
정답       : 먹고
현재 예측   : 나는

현재 Hidden State
[-0.01546194 -0.00587204  0.01037285  0.01105312  0.01236869  0.00722006
 -0.00424484  0.0171963   0.00934141 -0.01234856  0.00339003  0.01327452
 -0.00704131  0.00428275 -0.00455192  0.00251008  0.00976916 -0.00548607]
Time Step : 3
입력       : 먹고
정답       : 싶다
현재 예측   : 싶다

현재 Hidden State
[ 0.0062841   0.00509885  0.00313531 -0.00960931 -0.00387108 -0.01305629
 -0.00517021 -0.0065852  -0.0070289   0.01262519 -0.00615148 -0.0020913
 -0.00044233 -0.00260697  0.00070489  0.01996974 -0.01703144 -0.00083438]


In [ ]:
from datasets import load_dataset
from kiwipiepy import Kiwi
from collections import Counter

dataset = load_dataset("FISA-conclave/news-sentiment-dataset")

kiwi = Kiwi()
counter = Counter()

for text in dataset["train"]["sentence"]:
    tokens = kiwi.tokenize(text)

    for token in tokens:
        counter[token.form] += 1

print(counter.most_common(20))

VOCAB_SIZE = 20000

word2idx = {"<PAD>": 0, "<UNK>": 1}

for word, count in counter.most_common(VOCAB_SIZE - 2):
    word2idx[word] = len(word2idx)


def text_to_indices(text):
    tokens = kiwi.tokenize(text)

    indices = []

    for token in tokens:
        word = token.form

        if word in word2idx:
            indices.append(word2idx[word])
        else:
            indices.append(word2idx["<UNK>"])
    return indices

[('하', 142497), ('다', 83141), ('을', 82981), ('이', 82188), ('ᆫ', 72156), ('는', 66799), ('었', 63428), ('.', 58801), (',', 56032), ('의', 50097), ('은', 49128), ('어', 48615), ('에', 48115), ('를', 47944), ('고', 35624), ('으로', 33897), ('(', 31554), (')', 31549), ("'", 27602), ('되', 26948)]
temp_text :  황규선 DL이앤씨 기획관리실장은 ㈜대림 최고운영책임자(COO)로 발탁됐다.

토큰:
['황', '규', '선', 'DL', '이앤씨', '기획', '관리실장', '은', '㈜', '대림', '최고운영책임자', '(', 'COO', ')', '로', '발탁', '되', '었', '다', '.']

숫자:
[2088, 1236, 650, 344, 414, 442, 3135, 12, 448, 1986, 2691, 18, 2642, 19, 29, 1055, 21, 8, 3, 9]


In [36]:
import torch
from torch.utils.data import Dataset

label2idx = {"negative": 0, "neutral": 1, "positive": 2}

idx2label = {0: "negative", 1: "neutral", 2: "positive"}


class NewsDataset(Dataset):
    def __init__(self, dataset_split):
        self.texts = dataset_split["sentence"]
        self.labels = dataset_split["label"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = text_to_indices(self.texts[idx])
        y = label2idx[self.labels[idx]]

        return x, y


train_dataset = NewsDataset(dataset["train"])
test_dataset = NewsDataset(dataset["test"])

X, y = train_dataset[0]
print("X : ", X)
print()
print("y : ", y)

X :  [2088, 1236, 650, 344, 414, 442, 3135, 12, 448, 1986, 2691, 18, 2642, 19, 29, 1055, 21, 8, 3, 9]

y :  1


In [ ]:
from torch.nn.utils.rnn import pad_sequence

PAD_IDX = word2idx["<PAD>"]


def collate_fn(batch):
    xs, ys = zip(*batch)

    xs = [torch.tensor(x, dtype=torch.long) for x in xs]

    xs = pad_sequence(xs, batch_first=True, padding_value=PAD_IDX)

    ys = torch.tensor(ys, dtype=torch.long)

    return xs, ys

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from datasets import load_dataset
from kiwipiepy import Kiwi
from collections import Counter


dataset = load_dataset("FISA-conclave/news-sentiment-dataset")

kiwi = Kiwi()

counter = Counter()

for text in dataset["train"]["sentence"]:
    tokens = kiwi.tokenize(text)

    for token in tokens:
        counter[token.form] += 1

전체 단어수 :  26583

자주 등장하는 단어 :  [('하', 142497), ('다', 83141), ('을', 82981), ('이', 82188), ('ᆫ', 72156), ('는', 66799), ('었', 63428), ('.', 58801), (',', 56032), ('의', 50097), ('은', 49128), ('어', 48615), ('에', 48115), ('를', 47944), ('고', 35624), ('으로', 33897), ('(', 31554), (')', 31549), ("'", 27602), ('되', 26948)]


In [ ]:
vocab_size = len(counter)

word2idx = {"<PAD>": 0, "<UNK>": 1}

for word, count in counter.most_common(vocab_size - 2):
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}


def text_to_indices(text):
    tokens = kiwi.tokenize(text)

    indices = []

    for token in tokens:
        word = token.form

        if word in word2idx:
            indices.append(word2idx[word])
        else:
            indices.append(word2idx["<UNK>"])
    return indices


label2idx = {"negative": 0, "neutral": 1, "positive": 2}

idx2label = {0: "negative", 1: "neutral", 2: "positive"}


class NewsDataset(Dataset):
    def __init__(self, dataset_split):
        self.texts = dataset_split["sentence"]
        self.labels = dataset_split["label"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = text_to_indices(self.texts[idx])

        y = label2idx[self.labels[idx]]

        return x, y


train_dataset = NewsDataset(dataset["train"])

test_dataset = NewsDataset(dataset["test"])

PAD_IDX = word2idx["<PAD>"]


def collate_fn(batch):
    xs, ys = zip(*batch)

    xs = [torch.tensor(x, dtype=torch.long) for x in xs]

    xs = pad_sequence(xs, batch_first=True, padding_value=PAD_IDX)

    ys = torch.tensor(ys, dtype=torch.long)

    return xs, ys


train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn
)


X shape :  torch.Size([64, 492])
y shape :  torch.Size([64])


In [ ]:
import torch
import torch.nn as nn


class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=PAD_IDX
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)

        output, hidden = self.rnn(x)

        last_hidden = hidden[-1]

        logits = self.fc(last_hidden)

        return logits


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("사용 장치:", device)

VOCAB_SIZE = len(word2idx)

EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_CLASSES = 3

model = RNNClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
)

model = model.to(device)

사용 장치: mps
RNNClassifier(
  (embedding): Embedding(26583, 128, padding_idx=0)
  (rnn): RNN(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)


In [2]:
import torch.nn as nn
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from datasets import load_dataset
from kiwipiepy import Kiwi
from collections import Counter


dataset = load_dataset("FISA-conclave/news-sentiment-dataset")

kiwi = Kiwi()

counter = Counter()

for text in dataset["train"]["sentence"]:
    tokens = kiwi.tokenize(text)

    for token in tokens:
        counter[token.form] += 1

vocab_size = len(counter)

word2idx = {"<PAD>": 0, "<UNK>": 1}

for word, count in counter.most_common(vocab_size - 2):
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}


def text_to_indices(text):
    tokens = kiwi.tokenize(text)

    indices = []

    for token in tokens:
        word = token.form

        if word in word2idx:
            indices.append(word2idx[word])
        else:
            indices.append(word2idx["<UNK>"])
    return indices


label2idx = {"negative": 0, "neutral": 1, "positive": 2}

idx2label = {0: "negative", 1: "neutral", 2: "positive"}


class NewsDataset(Dataset):
    def __init__(self, dataset_split):
        self.texts = dataset_split["sentence"]
        self.labels = dataset_split["label"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = text_to_indices(self.texts[idx])

        y = label2idx[self.labels[idx]]

        return x, y


train_dataset = NewsDataset(dataset["train"])

test_dataset = NewsDataset(dataset["test"])

PAD_IDX = word2idx["<PAD>"]


def collate_fn(batch):
    xs, ys = zip(*batch)

    xs = [torch.tensor(x, dtype=torch.long) for x in xs]

    xs = pad_sequence(xs, batch_first=True, padding_value=PAD_IDX)

    ys = torch.tensor(ys, dtype=torch.long)

    return xs, ys


train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn
)


class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=PAD_IDX
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)

        output, hidden = self.rnn(x)

        last_hidden = hidden[-1]

        logits = self.fc(last_hidden)

        return logits


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("사용 장치:", device)

VOCAB_SIZE = len(word2idx)

EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_CLASSES = 3

model = RNNClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
)

model = model.to(device)

EPOCHS = 5

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(x_batch)

        loss = criterion(logits, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = logits.argmax(dim=1)

        correct += (predictions == y_batch).sum().item()

        total += y_batch.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total

    print(f"Epoch [{epoch + 1}/{EPOCHS}] Loss: {avg_loss:.4f} Accuracy: {accuracy:.4f}")

사용 장치: mps
Epoch [1/5] Loss: 0.9313 Accuracy: 0.5124
Epoch [2/5] Loss: 0.9333 Accuracy: 0.5112
Epoch [3/5] Loss: 0.9424 Accuracy: 0.4964
Epoch [4/5] Loss: 0.9347 Accuracy: 0.5013
Epoch [5/5] Loss: 0.9406 Accuracy: 0.5047


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# 1. 아주 작은 번역 데이터
# ============================================================

pairs = [
    ("나는 먹는다", "i eat"),
    ("나는 잔다", "i sleep"),
    ("나는 달린다", "i run"),
    ("너는 먹는다", "you eat"),
    ("너는 잔다", "you sleep"),
    ("너는 달린다", "you run"),
]


# ============================================================
# 2. 한국어 / 영어 Vocabulary 만들기
# ============================================================

korean_vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "나는": 3,
    "너는": 4,
    "먹는다": 5,
    "잔다": 6,
    "달린다": 7,
}

english_vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "i": 3,
    "you": 4,
    "eat": 5,
    "sleep": 6,
    "run": 7,
}

english_idx2word = {value: key for key, value in english_vocab.items()}


# ============================================================
# 3. 문장을 Token ID로 변환
# ============================================================


def korean_to_ids(sentence):
    tokens = sentence.split()

    return [korean_vocab[token] for token in tokens]


def english_to_ids(sentence):
    tokens = sentence.split()

    return [
        english_vocab["<SOS>"],
        *[english_vocab[token] for token in tokens],
        english_vocab["<EOS>"],
    ]


# ============================================================
# 4. Encoder
# ============================================================


class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):

        super().__init__()

        # 한국어 Token ID → 한국어 벡터
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 한국어 벡터 → hidden state
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

    def forward(self, x):

        # x:
        # [한국어 Token ID]
        #
        # 예:
        # [3, 5]
        #
        # 나는 = 3
        # 먹는다 = 5

        embedded = self.embedding(x)

        # embedded:
        # [2개의 단어, embedding_dim]

        outputs, hidden = self.rnn(embedded)

        # hidden:
        # Encoder가 한국어 문장을 읽고
        # 마지막에 만든 기억

        return hidden


# ============================================================
# 5. Decoder
# ============================================================


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):

        super().__init__()

        # 영어 Token ID → 영어 벡터
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 영어 벡터 + Encoder의 hidden
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

        # hidden → 영어 단어 점수
        #
        # 영어 단어가 8개이므로
        # 출력도 8개
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):

        # x:
        # 현재 Decoder에 들어가는 영어 단어
        #
        # 예:
        # <SOS>

        embedded = self.embedding(x)

        output, hidden = self.rnn(embedded, hidden)

        # hidden state를
        # 영어 단어 8개의 점수로 변환

        prediction = self.fc(output)

        return prediction, hidden


# ============================================================
# 6. Encoder + Decoder
# ============================================================

embedding_dim = 16
hidden_dim = 32

encoder = Encoder(
    vocab_size=len(korean_vocab), embedding_dim=embedding_dim, hidden_dim=hidden_dim
)

decoder = Decoder(
    vocab_size=len(english_vocab), embedding_dim=embedding_dim, hidden_dim=hidden_dim
)


# ============================================================
# 7. Loss / Optimizer
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)


# ============================================================
# 8. 학습
# ============================================================

for epoch in range(1000):
    total_loss = 0

    for korean, english in pairs:
        # ----------------------------------------------------
        # 한국어
        # ----------------------------------------------------

        korean_ids = korean_to_ids(korean)

        korean_tensor = torch.tensor(korean_ids, dtype=torch.long).unsqueeze(0)

        # 예:
        #
        # "나는 먹는다"
        #
        # ↓
        #
        # [3, 5]

        # ----------------------------------------------------
        # 영어
        # ----------------------------------------------------

        english_ids = english_to_ids(english)

        english_tensor = torch.tensor(english_ids, dtype=torch.long).unsqueeze(0)

        # 예:
        #
        # "i eat"
        #
        # ↓
        #
        # [<SOS>, i, eat, <EOS>]

        # ----------------------------------------------------
        # Encoder
        # ----------------------------------------------------

        hidden = encoder(korean_tensor)

        # 한국어
        #
        # 나는 → 먹는다
        #
        # ↓
        #
        # Encoder
        #
        # ↓
        #
        # hidden

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        decoder_input = english_tensor[:, 0:1]
        # 처음에는 <SOS>

        loss = 0

        for t in range(1, english_tensor.size(1)):
            output, hidden = decoder(decoder_input, hidden)

            # output:
            #
            # 영어 단어 8개의 점수
            #
            # [<PAD>, <SOS>, <EOS>, i, you, eat, sleep, run]

            target = english_tensor[:, t]

            # 정답 영어 단어

            loss += criterion(output.squeeze(1), target)

            # 학습할 때는
            # 정답 단어를 다음 입력으로 사용

            decoder_input = target.unsqueeze(1)

        # ----------------------------------------------------
        # 역전파
        # ----------------------------------------------------

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")


# ============================================================
# 9. 번역 함수
# ============================================================


def translate(sentence):

    encoder.eval()
    decoder.eval()

    with torch.no_grad():
        # ----------------------------------------------------
        # 한국어 → Token ID
        # ----------------------------------------------------

        korean_ids = korean_to_ids(sentence)

        korean_tensor = torch.tensor(korean_ids, dtype=torch.long).unsqueeze(0)

        # ----------------------------------------------------
        # Encoder
        # ----------------------------------------------------

        hidden = encoder(korean_tensor)

        # ----------------------------------------------------
        # Decoder 시작
        # ----------------------------------------------------

        decoder_input = torch.tensor([[english_vocab["<SOS>"]]], dtype=torch.long)

        result = []

        # 최대 10개의 영어 단어 생성

        for _ in range(10):
            output, hidden = decoder(decoder_input, hidden)

            # 가장 높은 점수를 가진 단어 선택

            next_token = output.argmax(dim=-1).item()

            # EOS면 종료

            if next_token == english_vocab["<EOS>"]:
                break

            # 영어 단어로 변환

            word = english_idx2word[next_token]

            result.append(word)

            # 방금 만든 단어를
            # 다음 입력으로 사용

            decoder_input = torch.tensor([[next_token]], dtype=torch.long)

        return " ".join(result)


# ============================================================
# 10. 실제 번역 테스트
# ============================================================

print()
print("번역 결과")
print("--------------------")

test_sentences = [
    "나는 먹는다",
    "나는 잔다",
    "나는 달린다",
    "너는 먹는다",
    "너는 잔다",
    "너는 달린다",
]

for sentence in test_sentences:
    result = translate(sentence)

    print(f"{sentence} → {result}")

english_tensor :  tensor([[1, 3, 5, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 3, 6, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 3, 7, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 4, 5, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 4, 6, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 4, 7, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 3, 5, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 3, 6, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 3, 7, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 4, 5, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 4, 6, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 4, 7, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 3, 5, 2]])
decoder_input :  tensor([[1]])
english_tensor :  tensor([[1, 3, 6, 2]])
decoder_input :  tensor